In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "enviroment_bj").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("project_root:", ROOT)


project_root: C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl


In [6]:
from enviroment_bj import BlackjackJSONWrapper, BlackjackConfig

wrapper = BlackjackJSONWrapper(
    config=BlackjackConfig(
        n_decks=6,
        shoe_penetration=0.8,
        base_bet=1.0,
    ),
    seed=42,
)

state = wrapper.reset()
state


{'observation': {'round_index': 1,
  'round_over': False,
  'round_reward': 0.0,
  'current_hand_index': 0,
  'current_hand': {'index': 0,
   'cards': ['6', '3'],
   'total': 9,
   'is_soft': False,
   'is_blackjack': False,
   'is_bust': False,
   'bet': 1.0,
   'doubled': False,
   'from_split': False,
   'split_aces': False,
   'closed': False,
   'surrendered': False,
   'action_count': 0,
   'close_reason': None,
   'settlement': None,
   'reward': 0.0},
  'player_hands': [{'index': 0,
    'cards': ['6', '3'],
    'total': 9,
    'is_soft': False,
    'is_blackjack': False,
    'is_bust': False,
    'bet': 1.0,
    'doubled': False,
    'from_split': False,
    'split_aces': False,
    'closed': False,
    'surrendered': False,
    'action_count': 0,
    'close_reason': None,
    'settlement': None,
    'reward': 0.0}],
  'dealer': {'upcard': 'J',
   'cards': ['J'],
   'hole_card_hidden': True,
   'visible_total': 10,
   'visible_is_soft': False,
   'peek_checked': True,
   'has_b

In [7]:
obs = state["observation"]

print("round_index:", obs["round_index"])
print("legal_actions:", obs["legal_actions"])
print("current_hand:", obs["current_hand"])
print("dealer:", obs["dealer"])
print("agent_state:", obs["agent_state"])
print("action_mask:", obs["action_mask"])
print("action_mask_by_name:", obs["action_mask_by_name"])


round_index: 1
legal_actions: ['stand', 'hit', 'double', 'surrender']
current_hand: {'index': 0, 'cards': ['6', '3'], 'total': 9, 'is_soft': False, 'is_blackjack': False, 'is_bust': False, 'bet': 1.0, 'doubled': False, 'from_split': False, 'split_aces': False, 'closed': False, 'surrendered': False, 'action_count': 0, 'close_reason': None, 'settlement': None, 'reward': 0.0}
dealer: {'upcard': 'J', 'cards': ['J'], 'hole_card_hidden': True, 'visible_total': 10, 'visible_is_soft': False, 'peek_checked': True, 'has_blackjack': None}
agent_state: {'player_total': 9, 'player_is_soft': False, 'dealer_upcard': 'J', 'dealer_upcard_value': 10, 'pair_for_split': False, 'can_double': True, 'can_hit': True, 'can_stand': True, 'can_surrender': True, 'insurance_offer_active': False, 'n_player_hands': 1, 'current_bet': 1.0, 'current_hand_index': 0, 'from_split': False, 'split_aces': False, 'dealer_blackjack_checked': True}
action_mask: [1, 1, 1, 0, 1, 0]
action_mask_by_name: {'stand': True, 'hit': True

In [8]:
# forma más simple
state = wrapper.step({"action": "hit"})

# también sirven estas formas:
# state = wrapper.step("hit")
# state = wrapper.step(1)   # 1 == "hit"
# state = wrapper.step({"hit": 1, "stand": 0})

state


{'observation': {'round_index': 1,
  'round_over': False,
  'round_reward': 0.0,
  'current_hand_index': 0,
  'current_hand': {'index': 0,
   'cards': ['6', '3', '9'],
   'total': 18,
   'is_soft': False,
   'is_blackjack': False,
   'is_bust': False,
   'bet': 1.0,
   'doubled': False,
   'from_split': False,
   'split_aces': False,
   'closed': False,
   'surrendered': False,
   'action_count': 1,
   'close_reason': None,
   'settlement': None,
   'reward': 0.0},
  'player_hands': [{'index': 0,
    'cards': ['6', '3', '9'],
    'total': 18,
    'is_soft': False,
    'is_blackjack': False,
    'is_bust': False,
    'bet': 1.0,
    'doubled': False,
    'from_split': False,
    'split_aces': False,
    'closed': False,
    'surrendered': False,
    'action_count': 1,
    'close_reason': None,
    'settlement': None,
    'reward': 0.0}],
  'dealer': {'upcard': 'J',
   'cards': ['J'],
   'hole_card_hidden': True,
   'visible_total': 10,
   'visible_is_soft': False,
   'peek_checked': Tru

In [9]:
print("reward:", state["reward"])
print("terminated:", state["terminated"])
print("info:", state["info"])
print("mano actual:", state["observation"]["current_hand"])
print("acciones legales:", state["observation"]["legal_actions"])


reward: 0.0
terminated: False
info: {'action': 'hit', 'round_index': 1, 'round_over': False, 'round_reward': 0.0, 'insurance_reward': 0.0, 'hand_rewards': [0.0], 'hand_settlements': [None]}
mano actual: {'index': 0, 'cards': ['6', '3', '9'], 'total': 18, 'is_soft': False, 'is_blackjack': False, 'is_bust': False, 'bet': 1.0, 'doubled': False, 'from_split': False, 'split_aces': False, 'closed': False, 'surrendered': False, 'action_count': 1, 'close_reason': None, 'settlement': None, 'reward': 0.0}
acciones legales: ['stand', 'hit']


In [10]:
state = wrapper.reset()

while not state["terminated"]:
    obs = state["observation"]
    hand = obs["current_hand"]
    legal = obs["legal_actions"]

    if "double" in legal and hand["total"] in (10, 11):
        action = "double"
    elif "hit" in legal and hand["total"] < 17:
        action = "hit"
    else:
        action = "stand"

    print("accion:", action)
    state = wrapper.step({"action": action})

print("round_reward:", state["reward"])
print("dealer final:", state["observation"]["dealer"])
print("player_hands:", state["observation"]["player_hands"])
print("info final:", state["info"])


accion: hit
round_reward: -1.0
dealer final: {'upcard': '3', 'cards': ['3', '2'], 'hole_card_hidden': False, 'visible_total': 5, 'visible_is_soft': False, 'peek_checked': False, 'has_blackjack': False, 'total': 5, 'is_soft': False}
player_hands: [{'index': 0, 'cards': ['Q', '3', 'Q'], 'total': 23, 'is_soft': False, 'is_blackjack': False, 'is_bust': True, 'bet': 1.0, 'doubled': False, 'from_split': False, 'split_aces': False, 'closed': True, 'surrendered': False, 'action_count': 1, 'close_reason': 'bust', 'settlement': 'loss', 'reward': -1.0}]
info final: {'action': 'hit', 'round_index': 2, 'round_over': True, 'round_reward': -1.0, 'insurance_reward': 0.0, 'hand_rewards': [-1.0], 'hand_settlements': ['loss']}


In [ ]:
# si terminated == True, la siguiente mano empieza con reset()
next_state = wrapper.reset()

print("nueva ronda:", next_state["observation"]["round_index"])
print("nueva mano:", next_state["observation"]["current_hand"])


In [11]:
from enviroment_bj import BlackjackJSONWrapper, BlackjackConfig

wrapper = BlackjackJSONWrapper(
    config=BlackjackConfig(n_decks=1, shoe_penetration=1.0),
    seed=11,
)

# orden forzado del shoe para que el ejemplo sea reproducible
wrapper.environment.load_shoe(
    ["10", "6", "7", "10", "10", "9", "5", "2", "10", "K", "8"],
    total_cards=11,
)

state = wrapper.reset()
print(state["observation"]["current_hand"])   # ['10', '7']
print(state["observation"]["dealer"])         # upcard 6

state = wrapper.step({"action": "stand"})
print(state["terminated"])                    # True
print(state["reward"])                        # 1.0
print(state["info"])

state = wrapper.reset()
print(state["observation"]["current_hand"])   # siguiente mano del mismo shoe


{'index': 0, 'cards': ['10', '7'], 'total': 17, 'is_soft': False, 'is_blackjack': False, 'is_bust': False, 'bet': 1.0, 'doubled': False, 'from_split': False, 'split_aces': False, 'closed': False, 'surrendered': False, 'action_count': 0, 'close_reason': None, 'settlement': None, 'reward': 0.0}
{'upcard': '6', 'cards': ['6'], 'hole_card_hidden': True, 'visible_total': 6, 'visible_is_soft': False, 'peek_checked': False, 'has_blackjack': None}
True
1.0
{'action': 'stand', 'round_index': 1, 'round_over': True, 'round_reward': 1.0, 'insurance_reward': 0.0, 'hand_rewards': [1.0], 'hand_settlements': ['win']}
{'index': 0, 'cards': ['9', '2'], 'total': 11, 'is_soft': False, 'is_blackjack': False, 'is_bust': False, 'bet': 1.0, 'doubled': False, 'from_split': False, 'split_aces': False, 'closed': False, 'surrendered': False, 'action_count': 0, 'close_reason': None, 'settlement': None, 'reward': 0.0}


---

In [2]:
from enviroment_bj import BlackjackTextGame, BlackjackConfig

game = BlackjackTextGame(
    config=BlackjackConfig(n_decks=1, shoe_penetration=1.0),
    seed=42,)


In [3]:
game.new_round()


Round 1
Dealer: J ? (10)
-> You[0]: 10 K (20)
Actions: stand, hit, double, split, surrender

In [4]:
game.stand()

Round 1
Dealer: J 4 9 (23)
   You[0]: 10 K (20) | win, +1.00
Round result: +1.00
Use game.new_round() to play again.

In [5]:
game.new_round()

Round 2
Dealer: 4 7 (11)
-> You[0]: K A (21) | soft, blackjack, +1.50
Round result: +1.50
Use game.new_round() to play again.

In [6]:
game.new_round()

Round 3
Dealer: 8 ? (8)
-> You[0]: Q Q (20)
Actions: stand, hit, double, split, surrender

In [7]:
game.status()

Round 3
Dealer: 8 ? (8)
-> You[0]: Q Q (20)
Actions: stand, hit, double, split, surrender